# Lab 007 — Simulate False Discoveries Under Many Trials

**Lesson:** [`0007-hypothesis-multiple-testing.html`](../lessons/0007-hypothesis-multiple-testing.html)

**The one skill:** show yourself, with a simulation, that testing *many* zero-edge strategies manufactures "significant" winners by pure chance — and that **Bonferroni** (family-wise) and **Benjamini-Hochberg** (false-discovery-rate) corrections put them back in the box. Then measure the **selection bias** of reporting the single best backtest.

**Exit criteria:** the EXIT TICKET cell prints cleanly (all CHECK cells pass).

**How this notebook works**

| Cell tag | You do |
|----------|--------|
| **PROVIDED** | Run it. Imports, data, helpers. |
| **TODO** | Fill the `____` blanks. This is where the learning is. |
| **CHECK** | Run it — immediate assertions. Don't edit. |
| **EXIT TICKET** | Final deliverable. Prints your findings. |

**Environment:** Python 3 + `numpy`, `scipy`. See [`labs/README.md`](./README.md).

## Concept recap (read before coding)

**t-statistic of a return series** ($n$ periods, mean $\hat\mu$, std $\hat\sigma$), testing $H_0:\mu=0$:
$$t = \frac{\hat\mu}{\hat\sigma/\sqrt{n}} = \frac{\hat\mu}{\hat\sigma}\sqrt{n} = SR_{\text{period}}\sqrt{n}.$$

**p-value (two-sided):** $p = 2\,[1 - \Phi(|t|)]$ using the standard-normal CDF $\Phi$ (large-$n$ approximation; `scipy.stats.norm.sf` gives the tail directly).

**Under $H_0$ the p-value is Uniform(0,1)** — so $P(p<\alpha)=\alpha$ per test.

**Family-wise error rate (independent nulls):** $\text{FWER}=1-(1-\alpha)^M$.

**Bonferroni:** reject test $i$ iff $p_i < \alpha/M$ → FWER $\le \alpha$.

**Benjamini-Hochberg (target FDR $q$):** sort $p_{(1)}\le\dots\le p_{(M)}$; find the largest $k$ with $p_{(k)}\le \frac{k}{M}q$; reject ranks $1..k$.

**Toy example (not the lab's answer):** 3 p-values `[0.004, 0.03, 0.20]`, $q=0.10$, thresholds $\frac{k}{3}0.10=[0.033,0.067,0.10]$: rank 1 ✓ (0.004≤0.033), rank 2 ✓ (0.03≤0.067), rank 3 ✗ → reject the first two.

In [ ]:
# PROVIDED — imports and helpers. No real market data needed: the whole point is a controlled
# universe where we KNOW the truth (zero edge), so any 'discovery' is provably false.
import numpy as np
from scipy import stats

ALPHA = 0.05        # single-test significance level
N = 252             # trading days per simulated strategy (~1 year)
SIGMA = 0.01        # daily vol of every strategy's returns

def t_stat(returns):
    """t-statistic for H0: mean == 0, per column of a (n_days, n_strats) array."""
    returns = np.asarray(returns, dtype=float)
    mu = returns.mean(axis=0)
    sd = returns.std(axis=0, ddof=1)
    n = returns.shape[0]
    return mu / (sd / np.sqrt(n))

def two_sided_p(t):
    """Two-sided p-value from a t-stat using the large-n normal approximation."""
    return 2.0 * stats.norm.sf(np.abs(t))

rng = np.random.default_rng(7)
print(f"Setup: alpha={ALPHA}, N={N} days/strategy, daily sigma={SIGMA}. All strategies have TRUE edge = 0.")

### Task 1 — Build a universe of worthless strategies and test them
**Goal:** simulate `M = 200` strategies, each `N` days of pure-noise returns (mean 0), compute each strategy's t-stat and two-sided p-value, and count how many are 'significant' at `ALPHA`. Store `pvals` (length M) and the integer `n_false` = number with `p < ALPHA`.

**Why it matters:** every discovery here is *false by construction* — this is the false-discovery factory made real.

**Hint boundary:** `rets = rng.normal(0.0, SIGMA, size=(N, M))`; use the provided `t_stat` and `two_sided_p`; count with `(pvals < ALPHA).sum()`.

In [ ]:
# TODO — simulate M zero-edge strategies, test each, count false positives.
M = 200
rets = rng.normal(0.0, SIGMA, size=(N, M))   # (N days, M strategies), TRUE mean = 0
tstats = t_stat(rets)
pvals = ____                                  # two-sided p-values from tstats
n_false = int(____)                           # how many have p < ALPHA
print(f"{n_false} of {M} worthless strategies look 'significant' at {ALPHA} (expected ~{ALPHA*M:.0f}).")

In [ ]:
# CHECK — do not edit
assert pvals.shape == (M,), "pvals must have one entry per strategy"
assert np.all((pvals >= 0) & (pvals <= 1)), "p-values must lie in [0, 1]"
assert 0 <= n_false <= M, "n_false must be a count of strategies"
print("Task 1 OK — false positives at nominal 5%: %d" % n_false)

### Task 2 — Confirm the false-discovery rate and the FWER formula
**Goal:** repeat the whole experiment `TRIALS = 2000` times. Store `frac_sig` = the average fraction of strategies flagged significant per trial (should ≈ `ALPHA`), and `fwer_emp` = the fraction of trials with **at least one** false positive (should ≈ `1-(1-ALPHA)**M`).

**Why it matters:** this is the empirical proof that the single-test bar leaks ~5% per test and ~99% *somewhere* across 200 tests.

**Hint boundary:** loop `TRIALS` times; each trial: fresh `rng.normal(0, SIGMA, (N, M))`, p-values, `hits = (p < ALPHA).sum()`; accumulate `hits/M` and `hits >= 1`.

In [ ]:
# TODO — Monte Carlo the false-discovery rate and the FWER.
TRIALS = 2000
sig_fracs = np.empty(TRIALS)
any_false = np.empty(TRIALS, dtype=bool)
for i in range(TRIALS):
    r = rng.normal(0.0, SIGMA, size=(N, M))
    p = two_sided_p(t_stat(r))
    hits = (p < ALPHA).sum()
    sig_fracs[i] = hits / M
    any_false[i] = ____                        # True if this trial had >= 1 false positive
frac_sig = float(sig_fracs.mean())
fwer_emp = float(any_false.mean())
fwer_theory = 1 - (1 - ALPHA) ** M
print(f"Mean fraction significant : {frac_sig:.4f}  (target alpha = {ALPHA})")
print(f"Empirical FWER            : {fwer_emp:.4f}")
print(f"Theoretical 1-(1-a)^M     : {fwer_theory:.4f}")

In [ ]:
# CHECK — do not edit
assert abs(frac_sig - ALPHA) < 0.01, "mean significant fraction should sit near alpha"
assert abs(fwer_emp - fwer_theory) < 0.05, "empirical FWER should match 1-(1-alpha)^M"
print("Task 2 OK — false-discovery rate ~%.3f, FWER ~%.3f" % (frac_sig, fwer_emp))

### Task 3 — Apply Bonferroni and Benjamini-Hochberg
**Goal:** on the Task-1 `pvals`, count discoveries surviving each correction. Store `n_bonf` (p < ALPHA/M) and `n_bh` (Benjamini-Hochberg at `q = ALPHA`).

**Why it matters:** both corrections should wipe out the false discoveries here (there is no real edge to keep), which is exactly the guarantee they promise.

**Hint boundary:** Bonferroni is one comparison. For BH: `order = np.argsort(pvals)`; `ranks = np.arange(1, M+1)`; `thresh = ranks/M * ALPHA`; `passed = pvals[order] <= thresh`; if any pass, `k = np.max(np.where(passed)) + 1` else `k = 0`; `n_bh = k`.

In [ ]:
# TODO — Bonferroni and Benjamini-Hochberg on the Task-1 p-values.
n_bonf = int((pvals < ALPHA / M).sum())

order = np.argsort(pvals)
sorted_p = pvals[order]
ranks = np.arange(1, M + 1)
bh_thresh = ranks / M * ALPHA
passed = sorted_p <= bh_thresh
k = int(____)                                  # largest rank (1-based) that passes, else 0
n_bh = k
print(f"Survivors — Bonferroni: {n_bonf}   Benjamini-Hochberg (q={ALPHA}): {n_bh}   (both should be ~0)")

In [ ]:
# CHECK — do not edit
assert 0 <= n_bonf <= n_false, "Bonferroni cannot flag more than the uncorrected test"
assert 0 <= n_bh <= n_false, "BH cannot flag more than the uncorrected test"
print("Task 3 OK — corrections applied (Bonferroni=%d, BH=%d)" % (n_bonf, n_bh))

### Task 4 — The selection bias of reporting the best
**Goal:** across `TRIALS` fresh universes of `M` worthless strategies, record the **maximum** t-stat per universe (what you'd report if you picked the best). Store `max_ts` (length TRIALS) and `mean_max_t` = its average. Compare to the theoretical growth `sqrt(2*ln(M))`.

**Why it matters:** the best-of-M t-stat is biased far above 1.96 even though nothing is real — the mathematical core of data snooping.

**Hint boundary:** each trial: `t = t_stat(rng.normal(0, SIGMA, (N, M)))`; `max_ts[i] = np.max(t)`.

In [ ]:
# TODO — distribution of the best-of-M t-stat under the global null.
max_ts = np.empty(TRIALS)
for i in range(TRIALS):
    t = t_stat(rng.normal(0.0, SIGMA, size=(N, M)))
    max_ts[i] = ____                            # the largest t-stat in this universe
mean_max_t = float(max_ts.mean())
approx = np.sqrt(2 * np.log(M))
print(f"Mean best-of-{M} t-stat : {mean_max_t:.2f}   vs sqrt(2 ln M) = {approx:.2f}")
print(f"Fraction of 'best' strategies that clear 1.96: {(max_ts > 1.96).mean():.3f}")

In [ ]:
# CHECK — do not edit
assert max_ts.shape == (TRIALS,), "one max t-stat per trial"
assert mean_max_t > 1.96, "the best-of-M t-stat should sit well above the single-test bar"
print("Task 4 OK — mean best-of-M t-stat = %.2f (selection bias made visible)" % mean_max_t)

In [ ]:
# EXIT TICKET — paste this output to your teacher.
print("=== False-discovery report (all strategies have ZERO true edge) ===")
print(f"M strategies, N days           : {M}, {N}")
print(f"False positives at nominal 5%  : {n_false}  (expected {ALPHA*M:.0f})")
print(f"Mean false-discovery fraction  : {frac_sig:.4f}  (target alpha {ALPHA})")
print(f"Empirical vs theoretical FWER  : {fwer_emp:.3f} vs {fwer_theory:.3f}")
print(f"Survivors after Bonferroni     : {n_bonf}")
print(f"Survivors after Benjamini-Hochberg (q={ALPHA}): {n_bh}")
print(f"Mean best-of-M t-stat          : {mean_max_t:.2f}  (sqrt(2 ln M) = {approx:.2f})")
print()
print("One-sentence takeaway (edit me):")
print("Testing many worthless strategies manufactures ~5%-per-test false winners and a ~99% chance")
print("of at least one; correcting for the number of trials (Bonferroni/BH) removes them, and the")
print("best-of-M t-stat is biased above 1.96 by selection alone — so I must always report M.")

### Stretch (optional, ungraded)
- **Inject a few real edges.** Give 10 of the 200 strategies a small true daily mean (e.g. `+0.0007`). Now Bonferroni vs BH *differ*: measure each method's realized power (true edges kept) and false-discovery proportion — BH should keep more real edges at a controlled false-discovery share.
- **Correlated strategies.** Generate returns with a shared common factor so the M tests are dependent; show that plain Bonferroni is now too conservative and that a simulation-based (max-stat) threshold is tighter.
- **BHY under dependence.** Divide the BH thresholds by `c(M) = sum(1/i for i in 1..M)` and confirm it is stricter than BH.
- **Deflated-Sharpe preview (unit 055):** convert `mean_max_t` back to an implied 'best' Sharpe over the N-day window and see how much headline Sharpe pure selection can fabricate.